# IOAI — 2025 Selection Test Cv Segmentation Unet (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import torchvision, numpy as np
from PIL import Image
tr = torchvision.datasets.VOCSegmentation('voc', year='2012', image_set='train', download=True)
va = torchvision.datasets.VOCSegmentation('voc', year='2012', image_set='val', download=True)
SZ = 128
def conv(ds, n):
    I, M = [], []
    for i in range(min(n, len(ds))):
        im, m = ds[i]
        I.append(np.asarray(im.convert('RGB').resize((SZ, SZ), Image.BILINEAR), np.uint8))
        M.append(np.asarray(m.resize((SZ, SZ), Image.NEAREST), np.uint8))
    return np.stack(I), np.stack(M)
tri, trm = conv(tr, 500); vai, _ = conv(va, 200)
np.save('train_images.npy', tri); np.save('train_masks.npy', trm); np.save('val_images.npy', vai)
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Semantic Segmentation with U-Net (Pascal VOC 2012)

> **Singapore IOAI 2025 — Selection Test (CV)**

Assign each pixel one of 21 classes (background + 20 VOC objects) using a **U-Net**. Score = **mean IoU (mIoU)** over classes present in the validation masks. A self-contained 128×128 subset of VOC2012 is bundled (`train_images.npy`/`train_masks.npy`, `val_images.npy`); the validation masks are the hidden grading key. **Submit** scores `val_pred.npy` (`(N,128,128)` class ids) by mIoU. Baseline: a compact U-Net — deepen it / add augmentation to improve.

In [ ]:
import numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES, IGNORE = 21, 255
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

def load_x(p):
    x = np.load(p).astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return torch.tensor(x).permute(0, 3, 1, 2).contiguous()

Xtr = load_x('train_images.npy'); Ytr = torch.tensor(np.load('train_masks.npy').astype(np.int64))
Xva = load_x('val_images.npy')
Xtr.shape, Ytr.shape, Xva.shape

## Compact U-Net

In [ ]:
def cbr(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, c=21):
        super().__init__()
        self.d1 = cbr(3, 32); self.d2 = cbr(32, 64); self.d3 = cbr(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bott = cbr(128, 256)
        self.u3 = nn.ConvTranspose2d(256, 128, 2, 2); self.c3 = cbr(256, 128)
        self.u2 = nn.ConvTranspose2d(128, 64, 2, 2); self.c2 = cbr(128, 64)
        self.u1 = nn.ConvTranspose2d(64, 32, 2, 2); self.c1 = cbr(64, 32)
        self.out = nn.Conv2d(32, c, 1)
    def forward(self, x):
        e1 = self.d1(x); e2 = self.d2(self.pool(e1)); e3 = self.d3(self.pool(e2))
        b = self.bott(self.pool(e3))
        d3 = self.c3(torch.cat([self.u3(b), e3], 1))
        d2 = self.c2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.c1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

model = UNet(NUM_CLASSES).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
crit = nn.CrossEntropyLoss(ignore_index=IGNORE)

## mIoU helper + train (small held-out from train for self-check)

In [ ]:
def miou(pred, gt, n=NUM_CLASSES, ignore=IGNORE):
    ious = []
    for c in range(n):
        g = (gt == c) & (gt != ignore); p = (pred == c)
        if g.sum() == 0:
            continue
        inter = (g & p).sum(); union = (g | p).sum()
        ious.append(inter / union if union else 0.0)
    return float(np.mean(ious)) if ious else 0.0

cut = int(len(Xtr) * 0.85)
dl = DataLoader(TensorDataset(Xtr[:cut], Ytr[:cut]), batch_size=16, shuffle=True)

@torch.no_grad()
def predict(X, bs=32):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(model(X[i:i+bs].to(device)).argmax(1).cpu().numpy())
    return np.concatenate(out)

for epoch in range(60):
    model.train()
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    if (epoch + 1) % 10 == 0:
        vm = miou(predict(Xtr[cut:]), Ytr[cut:].numpy())
        print(f'epoch {epoch+1}  train-holdout mIoU {vm:.4f}')

## Predict the validation set → `val_pred.npy`

In [ ]:
pred = predict(Xva).astype(np.uint8)
np.save('val_pred.npy', pred)
print('wrote val_pred.npy', pred.shape)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['val_pred.npy']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)